In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from ira.ingest.ingest_scorecard import save
from ira.clean.clean_scorecard import clean
from ira.config import SCORECARD_KEY, BASE_URL
import math
import json
import time

root = Path.cwd()/'institutional-roi-analysis'
# os.chdir(root/"Seb_branch"/"institutional-roi-analysis"/"notebooks")
pd.set_option("display.max_columns",None)
display(root)

WindowsPath('C:/Users/sebas/PycharmProjects/Git/Seb_branch/institutional-roi-analysis')

### Feature Selection for Explanatory Model

Variables used in the initial prediction model (e.g., credential level, distance, and other structural constraints) were excluded from the explanatory model.

This is because the residuals already represent performance after controlling for these factors. Including them again would introduce circular reasoning and reduce the interpretability of the results.

Instead, the explanatory model focuses on institutional characteristics and program composition variables that were not used in the prediction stage, allowing us to better understand what drives over- and underperformance.

In [2]:
def get_with_retries(url, params, tries=5, timeout=30):
    last = None
    for i in range(tries):
        r = requests.get(
            url,
            params=params,
            timeout=timeout,
            headers={"Accept": "application/json"},
        )
        if r.status_code < 500 and r.status_code != 429:
            return r
        if r.status_code == 429:
            time.sleep(5 * (i + 1))
            continue
        last = r
        time.sleep((2 ** i) + random.random())
    return last


def get_json_or_raise(response: requests.Response):
    try:
        response.raise_for_status()
    except requests.HTTPError as e:
        ct = response.headers.get("Content-Type", "")
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"HTTP {response.status_code} for {response.url}\n"
            f"Content-Type: {ct}\n"
            f"Body preview:\n{body_preview}"
        ) from e

    ct = response.headers.get("Content-Type", "")
    if "json" not in ct.lower():
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"Expected JSON but got Content-Type: {ct}\n"
            f"URL: {response.url}\n"
            f"Body preview:\n{body_preview}"
        )

    try:
        return response.json()
    except json.JSONDecodeError as e:
        body_preview = (response.text or "")[:800]
        raise RuntimeError(
            f"JSON decode failed for {response.url}\n"
            f"Body preview:\n{body_preview}"
        ) from e


def collect_school_level(state: str = "FL", per_page: int = 100, api_key: str = "") -> pd.DataFrame:
    fields = ",".join([
        "id",
        "school.name",
        "unit"
        "latest.academics.program_percentage.agriculture",
        "latest.academics.program_percentage.resources",
        "latest.academics.program_percentage.architecture",
        "latest.academics.program_percentage.ethnic_cultural_gender",
        "latest.academics.program_percentage.communication",
        "latest.academics.program_percentage.communications_technology",
        "latest.academics.program_percentage.computer",
        "latest.academics.program_percentage.personal_culinary",
        "latest.academics.program_percentage.education",
        "latest.academics.program_percentage.engineering",
        "latest.academics.program_percentage.engineering_technology",
        "latest.academics.program_percentage.language",
        "latest.academics.program_percentage.family_consumer_science",
        "latest.academics.program_percentage.legal",
        "latest.academics.program_percentage.english",
        "latest.academics.program_percentage.humanities",
        "latest.academics.program_percentage.library",
        "latest.academics.program_percentage.biological",
        "latest.academics.program_percentage.mathematics",
        "latest.academics.program_percentage.military",
        "latest.academics.program_percentage.multidiscipline",
        "latest.academics.program_percentage.parks_recreation_fitness",
        "latest.academics.program_percentage.philosophy_religious",
        "latest.academics.program_percentage.theology_religious_vocation",
        "latest.academics.program_percentage.physical_science",
        "latest.academics.program_percentage.science_technology",
        "latest.academics.program_percentage.psychology",
        "latest.academics.program_percentage.security_law_enforcement",
        "latest.academics.program_percentage.public_administration_social_service",
        "latest.academics.program_percentage.social_science",
        "latest.academics.program_percentage.construction",
        "latest.academics.program_percentage.mechanic_repair_technology",
        "latest.academics.program_percentage.precision_production",
        "latest.academics.program_percentage.transportation",
        "latest.academics.program_percentage.visual_performing",
        "latest.academics.program_percentage.health",
        "latest.academics.program_percentage.business_marketing",
        "latest.academics.program_percentage.history",
        "latest.school.instructional_expenditure_per_fte",
        "latest.school.faculty_salary",
        "latest.school.ft_faculty_rate",
        "latest.academics.program_reporter.programs_offered",
        "latest.student.demographics.student_faculty_ratio",
        "latest.school.endowment.begin",
        "latest.school.endowment.end",
        "latest.school.dolflag",

    ])

    params = {
        "api_key": SCORECARD_KEY,
        "school.state": state,
        "fields": fields,
        "per_page": str(per_page),
        "page": "0",
    }

    response = get_with_retries(BASE_URL, params=params, timeout=30)
    data = get_json_or_raise(response)

    total = int(data["metadata"]["total"])
    per_page_actual = int(data["metadata"]["per_page"])
    total_pages = math.ceil(total / per_page_actual)

    rows = []

    for page in range(total_pages):
        params["page"] = str(page)
        response = get_with_retries(BASE_URL, params=params, timeout=30)
        data = get_json_or_raise(response)
        rows.extend(data.get("results", []))

    df = pd.json_normalize(rows)

    print(f"Total pages fetched: {total_pages}")
    print(f"Total schools ingested: {len(df)}")
    return df

In [3]:
tdf=collect_school_level()

Total pages fetched: 4
Total schools ingested: 374


In [4]:
tdf=clean(tdf)

Numeric columns: Index(['program_percentage_resources', 'program_percentage_architecture',
       'program_percentage_ethnic_cultural_gender',
       'program_percentage_communication',
       'program_percentage_communications_technology',
       'program_percentage_computer', 'program_percentage_personal_culinary',
       'program_percentage_education', 'program_percentage_engineering',
       'program_percentage_engineering_technology',
       'program_percentage_language',
       'program_percentage_family_consumer_science',
       'program_percentage_legal', 'program_percentage_english',
       'program_percentage_humanities', 'program_percentage_library',
       'program_percentage_biological', 'program_percentage_mathematics',
       'program_percentage_military', 'program_percentage_multidiscipline',
       'program_percentage_parks_recreation_fitness',
       'program_percentage_philosophy_religious',
       'program_percentage_theology_religious_vocation',
       'program_per

C:\Users\sebas\PycharmProjects\Git\Seb_branch\institutional-roi-analysis\src\ira\clean\clean_scorecard.py:8: FutureWarning: The default value of regex will change from True to False in a future version. In addition, single character regular expressions will *not* be treated as literal strings when regex=True.
  df.columns = df.columns.str.replace(r".", "_")


In [5]:
driver_df=tdf.copy()

In [6]:
driver_df["has_endowment"] = (
    driver_df["endowment_begin"].notna() &
    driver_df["endowment_end"].notna()
).astype(int)

In [7]:
driver_df["has_endowment"].describe()

count    374.000000
mean       0.200535
std        0.400937
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000
Name: has_endowment, dtype: float64

In [8]:
driver_df=driver_df.rename(columns={"id":"unit_id"})
save(driver_df,file_name="inst_driver")

In [9]:
driver_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 374 entries, 0 to 373
Data columns (total 48 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   program_percentage_resources                             306 non-null    float64
 1   program_percentage_architecture                          306 non-null    float64
 2   program_percentage_ethnic_cultural_gender                306 non-null    float64
 3   program_percentage_communication                         306 non-null    float64
 4   program_percentage_communications_technology             306 non-null    float64
 5   program_percentage_computer                              306 non-null    float64
 6   program_percentage_personal_culinary                     306 non-null    float64
 7   program_percentage_education                             306 non-null    float64
 8   program_percentage_engineering

In [58]:
residual_df = pd.read_csv(root/"data"/"raw"/"scorecard"/"raw_residual_FL_stable_programs.csv")
display(residual_df.info())
residual_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1367 entries, 0 to 1366
Data columns (total 54 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   code                       1367 non-null   int64  
 1   credential_level           1367 non-null   int64  
 2   unit_id                    1367 non-null   int64  
 3   distance                   1367 non-null   int64  
 4   school_type                1367 non-null   object 
 5   5_yr_working_count         1367 non-null   float64
 6   location_lat               1367 non-null   float64
 7   location_lon               1367 non-null   float64
 8   locale                     1367 non-null   int64  
 9   carnegie_size_setting      1367 non-null   int64  
 10  admission_rate_overall     866 non-null    float64
 11  median_family_income       1366 non-null   float64
 12  students_with_pell_grant   1304 non-null   float64
 13  open_admissions_policy     1366 non-null   float

None

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct,median_score
0,301,3,133492,1,"Private, nonprofit",56.0,27.715798,-82.687043,11,11,0.7580,77695.0,0.411392,2.0,21.0,1,open,10.662914,10.643175,Natural Resources Conservation and Research.,Eckerd College,42741.0,41905.617,835.382813,39932.0,10.594933,50566.960,10.831054,-10634.960938,77,25709.0,10.154596,29107.588,10.278754,-3398.587891,94.0,8,medium,-0.116760,-0.115420,-0.210314,-0.207225,0.019935,0.019527,6.0,8.0,3.0,2.0,-5.0,-3.0,2.516611,0.359516,0.250313,-0.115420
1,301,3,133951,2,Public,36.0,25.757320,-80.373928,21,15,0.5466,23924.0,0.813127,2.0,23.0,1,mid,10.745076,10.783957,Natural Resources Conservation and Research.,Florida International University,46401.0,48240.610,-1839.609375,49748.0,10.814726,55158.484,10.917966,-5410.484375,24,36007.0,10.491469,33721.535,10.425892,2285.464844,21.0,8,medium,0.067775,0.063452,-0.098090,-0.092620,-0.038134,-0.036835,2.0,7.0,6.0,5.0,-1.0,4.0,2.645751,0.377964,0.250313,-0.036835
2,301,3,134097,1,Public,78.0,30.443147,-84.295064,12,15,0.2422,43729.0,0.597532,2.0,20.0,1,elite,10.780705,10.786012,Natural Resources Conservation and Research.,Florida State University,48084.0,48339.850,-255.851562,54288.0,10.902058,52498.200,10.868534,1789.800781,188,30146.0,10.313808,36875.710,10.515308,-6729.710937,161.0,8,medium,-0.182497,-0.181380,0.034093,0.033916,-0.005293,-0.005219,7.0,5.0,5.0,-2.0,0.0,-2.0,1.154701,0.164957,0.250313,-0.005219
3,301,3,134130,1,Public,28.0,29.646290,-82.347911,12,16,0.2420,39127.0,0.667383,2.0,21.0,1,elite,10.895108,10.878085,Natural Resources Conservation and Research.,University of Florida,53912.0,53002.010,909.988281,62683.0,11.045846,57587.316,10.961058,5095.683594,29,34454.0,10.447380,36076.906,10.493408,-1622.906250,29.0,8,medium,-0.044985,-0.042980,0.088486,0.084491,0.017169,0.016391,5.0,3.0,4.0,-2.0,1.0,-1.0,1.000000,0.142857,0.250313,0.016391
4,301,3,136950,1,"Private, nonprofit",29.0,28.592787,-81.349239,21,11,0.4754,43978.0,0.580000,2.0,22.0,1,mid,10.517050,10.832608,Natural Resources Conservation and Research.,Rollins College,36940.0,50645.630,-13705.628906,63363.0,11.056635,52899.145,10.876143,10463.855469,23,22352.0,10.014671,34313.710,10.443300,-11961.710937,18.0,8,medium,-0.348599,-0.322316,0.197808,0.186247,-0.270618,-0.258835,8.0,2.0,8.0,-6.0,6.0,0.0,3.464102,0.494872,0.250313,-0.258835


In [59]:
join_cols=["unit_id"]
for col in join_cols:
    driver_df[col] = driver_df[col].astype(str).str.strip()
    residual_df[col] = residual_df[col].astype(str).str.strip()


### Target Variable Selection

While a composite scoring metric was developed to rank program-level variability, it was not used as the target for the explanatory model.

Instead, the model uses the average  raw percentage error of years 1, 4, and 5 as the target:

  * pct_error = error / predicted

This decision ensures that the model learns directly from observed over- and underperformance, rather than from a derived metric that incorporates additional adjustments (e.g., sample size penalties and variability scaling).

The composite score remains useful for identifying high-variability groups, but the explanatory model focuses on the underlying performance signal.

In [60]:
# residual_df["combined_pct_error"]=(
#     residual_df[['1_year_error','4_year_error','5_year_error']].sum(axis=1)
#     /
#     residual_df[['1_year_pred','4_year_pred','5_year_pred']].sum(axis=1)
# )

In [81]:
# compute pct error at row level in dollar space
residual_df["dollar_error_median"] = residual_df[
    ["1_year_error", "4_year_error", "5_year_error"]
].median(axis=1)

residual_df["dollar_pred_median"] = residual_df[
    ["1_year_pred", "4_year_pred", "5_year_pred"]
].median(axis=1)

residual_df["row_pct_error"] = (
   residual_df["dollar_error_median"] / residual_df["dollar_pred_median"]
)

In [82]:
school_df = residual_df.groupby("unit_id", as_index=False).agg(
    combined_pct_error=("row_pct_error", "median"),
    avg_rank_stability=("mean_rank_std_pct", "mean"),
    school_name=("school_name", "first"),
    total_count_1=("1_yr_working_count", "sum"),
    total_count_4=("4_yr_working_count", "sum"),
    total_count_5=("5_yr_working_count", "sum"),
)

school_df["total_count"] = (
    school_df["total_count_1"] +
    school_df["total_count_4"] +
    school_df["total_count_5"]
)

k = np.percentile(np.log1p(school_df["total_count"]), 75)
school_df["weight"] = (
    np.log1p(school_df["total_count"]) /
    np.log1p(school_df["total_count"] + k)
)

In [83]:
targ="combined_pct_error"
merge_df=driver_df.merge(school_df[[targ,'unit_id','weight']], on=["unit_id"],how="inner")
merge_df.head()

,program_percentage_resources,program_percentage_architecture,program_percentage_ethnic_cultural_gender,program_percentage_communication,program_percentage_communications_technology,program_percentage_computer,program_percentage_personal_culinary,program_percentage_education,program_percentage_engineering,program_percentage_engineering_technology,program_percentage_language,program_percentage_family_consumer_science,program_percentage_legal,program_percentage_english,program_percentage_humanities,program_percentage_library,program_percentage_biological,program_percentage_mathematics,program_percentage_military,program_percentage_multidiscipline,program_percentage_parks_recreation_fitness,program_percentage_philosophy_religious,program_percentage_theology_religious_vocation,program_percentage_physical_science,program_percentage_science_technology,program_percentage_psychology,program_percentage_security_law_enforcement,program_percentage_public_administration_social_service,program_percentage_social_science,program_percentage_construction,program_percentage_mechanic_repair_technology,program_percentage_precision_production,program_percentage_transportation,program_percentage_visual_performing,program_percentage_health,program_percentage_business_marketing,program_percentage_history,instructional_expenditure_per_fte,faculty_salary,ft_faculty_rate,program_reporter_programs_offered,student_faculty_ratio,endowment_begin,endowment_end,dolflag,school_name,unit_id,has_endowment,combined_pct_error,weight
0,0.0000,0.0,0.0,0.0366,0.0000,0.0539,0.0000,0.0173,0.0000,0.0000,0.0,0.0000,0.0077,0.0019,0.0135,0.0,0.1522,0.0019,0.0000,0.0000,0.0405,0.0019,0.0019,0.0096,0.0000,0.0636,0.0289,0.0829,0.0385,0.0,0.0000,0.0000,0.0000,0.0366,0.2216,0.1888,0.0,11546.0,8861.0,0.2830,NaN,7.0,53988781.0,58752320.0,0.0,Barry University,132471,1,-0.031149,0.999852
1,0.0034,0.0,0.0,0.1000,0.0000,0.0207,0.0000,0.0310,0.0138,0.0000,0.0,0.0000,0.0000,0.0069,0.2207,0.0,0.0448,0.0034,0.0000,0.0138,0.1000,0.0000,0.0000,0.0000,0.0000,0.1414,0.0793,0.0000,0.0310,0.0,0.0000,0.0000,0.0000,0.0172,0.0138,0.1586,0.0,11053.0,7167.0,0.7600,NaN,17.0,36303752.0,41110832.0,1.0,Bethune-Cookman University,132602,1,-0.141281,0.994868
2,0.0051,0.0,0.0,0.0696,0.0170,0.0000,0.0000,0.0357,0.0000,0.0000,0.0,0.0000,0.0000,0.0000,0.0017,0.0,0.0628,0.0000,0.0136,0.0068,0.0696,0.0000,0.0000,0.0000,0.0000,0.0951,0.0594,0.0000,0.0119,0.0,0.0000,0.0000,0.0611,0.1070,0.0051,0.3786,0.0,7523.0,6894.0,0.4331,NaN,16.0,37840050.0,42918486.0,0.0,Lynn University,132657,1,0.163185,0.998617
3,0.0000,0.0,0.0,0.0007,0.0019,0.0713,0.0110,0.0024,0.0000,0.0380,0.0,0.0129,0.0014,0.0000,0.3991,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0603,0.0000,0.0543,0.0000,0.0000,0.0,0.0115,0.0081,0.0000,0.0012,0.1706,0.1515,0.0,4778.0,6880.0,0.3651,NaN,22.0,19989561.0,22555029.0,1.0,Eastern Florida State College,132693,1,-0.006074,0.999767
4,0.0059,0.0,0.0,0.0014,0.0010,0.0992,0.0023,0.0092,0.0000,0.0119,0.0,0.0155,0.0028,0.0000,0.3970,0.0,0.0000,0.0000,0.0000,0.0025,0.0000,0.0000,0.0000,0.0000,0.0001,0.0000,0.0528,0.0000,0.0001,0.0,0.0035,0.0000,0.0261,0.0086,0.1196,0.2407,0.0,3883.0,6877.0,0.3117,NaN,31.0,44981606.0,45145927.0,1.0,Broward College,132709,1,0.020749,0.999803


In [84]:
model_df=merge_df.copy()
print("scorecard driver schools:", driver_df['unit_id'].nunique())
print("school_df school:", school_df['unit_id'].nunique())
print("model_df schools:", model_df['unit_id'].nunique())

print("model_df columns:")
print(sorted(model_df.columns.tolist()))

scorecard driver schools: 374
school_df school: 200
model_df schools: 200
model_df columns:
['combined_pct_error', 'dolflag', 'endowment_begin', 'endowment_end', 'faculty_salary', 'ft_faculty_rate', 'has_endowment', 'instructional_expenditure_per_fte', 'program_percentage_architecture', 'program_percentage_biological', 'program_percentage_business_marketing', 'program_percentage_communication', 'program_percentage_communications_technology', 'program_percentage_computer', 'program_percentage_construction', 'program_percentage_education', 'program_percentage_engineering', 'program_percentage_engineering_technology', 'program_percentage_english', 'program_percentage_ethnic_cultural_gender', 'program_percentage_family_consumer_science', 'program_percentage_health', 'program_percentage_history', 'program_percentage_humanities', 'program_percentage_language', 'program_percentage_legal', 'program_percentage_library', 'program_percentage_mathematics', 'program_percentage_mechanic_repair_techn

In [85]:
model_df=model_df.drop(columns=[
    # "program_reporter_programs_offered",
    # 'code',
    # 'unit_id',
    # 'school_name'
    ]
)

In [86]:
program_cols = [c for c in model_df.columns if c.startswith("program_percentage_")]

for c in program_cols:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce").fillna(0)

# STEM
model_df["pct_stem"] = model_df[
    [
        "program_percentage_computer",
        "program_percentage_engineering",
        "program_percentage_engineering_technology",
        "program_percentage_mathematics",
        "program_percentage_physical_science",
        "program_percentage_biological",
        "program_percentage_science_technology"
    ]
].sum(axis=1)

# Business / Econ
model_df["pct_business"] = model_df[
    ["program_percentage_business_marketing"]
].sum(axis=1)

# Health
model_df["pct_health"] = model_df[
    ["program_percentage_health"]
].sum(axis=1)

# Social Sciences
model_df["pct_social_science"] = model_df[
    [
        "program_percentage_psychology",
        "program_percentage_social_science",
        "program_percentage_history",
        "program_percentage_public_administration_social_service"
    ]
].sum(axis=1)

# Humanities
model_df["pct_humanities"] = model_df[
    [
        "program_percentage_english",
        "program_percentage_language",
        "program_percentage_humanities",
        "program_percentage_philosophy_religious",
        "program_percentage_theology_religious_vocation",
        "program_percentage_ethnic_cultural_gender"
    ]
].sum(axis=1)

# Arts & Communication
model_df["pct_arts_comm"] = model_df[
    [
        "program_percentage_visual_performing",
        "program_percentage_communication"
    ]
].sum(axis=1)

# Education
model_df["pct_education"] = model_df[
    ["program_percentage_education"]
].sum(axis=1)

# Trades / Technical
model_df["pct_trades"] = model_df[
    [
        "program_percentage_construction",
        "program_percentage_mechanic_repair_technology",
        "program_percentage_precision_production",
        "program_percentage_transportation"
    ]
].sum(axis=1)

# Services / Consumer
model_df["pct_services"] = model_df[
    [
        "program_percentage_personal_culinary",
        "program_percentage_family_consumer_science",
        "program_percentage_parks_recreation_fitness"
    ]
].sum(axis=1)

# Law / Security
model_df["pct_law_security"] = model_df[
    [
        "program_percentage_legal",
        "program_percentage_security_law_enforcement"
    ]
].sum(axis=1)

# Agriculture / Natural resources
model_df["pct_agriculture"] = model_df[
    [
        # "program_percentage_agriculture",
        "program_percentage_resources"
    ]
].sum(axis=1)

model_df["pct_high_roi"] = (
    model_df["program_percentage_engineering"] +
    model_df["program_percentage_computer"] +
    model_df["program_percentage_health"]
)

model_df["pct_low_roi"] = (
    model_df["program_percentage_education"] +
    model_df["program_percentage_personal_culinary"] +
    model_df["program_percentage_humanities"]
)

model_df["program_hhi"] = (model_df[program_cols] ** 2).sum(axis=1)

model_df["max_program_share"] = model_df[program_cols].max(axis=1)

model_df["high_roi_x_concentration"] = (
    model_df["pct_high_roi"] * model_df["program_hhi"]
)

# model_df = model_df.drop(columns=program_cols)



In [87]:
display("Repeated columns within instituions",(model_df.groupby("unit_id").nunique() > 1).sum())

'Repeated columns within instituions'

program_percentage_resources                    0
program_percentage_architecture                 0
program_percentage_ethnic_cultural_gender       0
program_percentage_communication                0
program_percentage_communications_technology    0
                                               ..
pct_high_roi                                    0
pct_low_roi                                     0
program_hhi                                     0
max_program_share                               0
high_roi_x_concentration                        0
Length: 65, dtype: int64

In [88]:
inst_model_df = model_df.drop_duplicates(subset="unit_id").reset_index(drop=True)
print("inst_model_df shape:", inst_model_df.shape)
display(inst_model_df.head())

inst_model_df shape: (200, 66)


,program_percentage_resources,program_percentage_architecture,program_percentage_ethnic_cultural_gender,program_percentage_communication,program_percentage_communications_technology,program_percentage_computer,program_percentage_personal_culinary,program_percentage_education,program_percentage_engineering,program_percentage_engineering_technology,program_percentage_language,program_percentage_family_consumer_science,program_percentage_legal,program_percentage_english,program_percentage_humanities,program_percentage_library,program_percentage_biological,program_percentage_mathematics,program_percentage_military,program_percentage_multidiscipline,program_percentage_parks_recreation_fitness,program_percentage_philosophy_religious,program_percentage_theology_religious_vocation,program_percentage_physical_science,program_percentage_science_technology,program_percentage_psychology,program_percentage_security_law_enforcement,program_percentage_public_administration_social_service,program_percentage_social_science,program_percentage_construction,program_percentage_mechanic_repair_technology,program_percentage_precision_production,program_percentage_transportation,program_percentage_visual_performing,program_percentage_health,program_percentage_business_marketing,program_percentage_history,instructional_expenditure_per_fte,faculty_salary,ft_faculty_rate,program_reporter_programs_offered,student_faculty_ratio,endowment_begin,endowment_end,dolflag,school_name,unit_id,has_endowment,combined_pct_error,weight,pct_stem,pct_business,pct_health,pct_social_science,pct_humanities,pct_arts_comm,pct_education,pct_trades,pct_services,pct_law_security,pct_agriculture,pct_high_roi,pct_low_roi,program_hhi,max_program_share,high_roi_x_concentration
0,0.0000,0.0,0.0,0.0366,0.0000,0.0539,0.0000,0.0173,0.0000,0.0000,0.0,0.0000,0.0077,0.0019,0.0135,0.0,0.1522,0.0019,0.0000,0.0000,0.0405,0.0019,0.0019,0.0096,0.0000,0.0636,0.0289,0.0829,0.0385,0.0,0.0000,0.0000,0.0000,0.0366,0.2216,0.1888,0.0,11546.0,8861.0,0.2830,NaN,7.0,53988781.0,58752320.0,0.0,Barry University,132471,1,-0.031149,0.999852,0.2176,0.1888,0.2216,0.1850,0.0192,0.0732,0.0173,0.0000,0.0405,0.0366,0.0000,0.2755,0.0308,0.129024,0.2216,0.035546
1,0.0034,0.0,0.0,0.1000,0.0000,0.0207,0.0000,0.0310,0.0138,0.0000,0.0,0.0000,0.0000,0.0069,0.2207,0.0,0.0448,0.0034,0.0000,0.0138,0.1000,0.0000,0.0000,0.0000,0.0000,0.1414,0.0793,0.0000,0.0310,0.0,0.0000,0.0000,0.0000,0.0172,0.0138,0.1586,0.0,11053.0,7167.0,0.7600,NaN,17.0,36303752.0,41110832.0,1.0,Bethune-Cookman University,132602,1,-0.141281,0.994868,0.0827,0.1586,0.0138,0.1724,0.2276,0.1172,0.0310,0.0000,0.1000,0.0793,0.0034,0.0483,0.2517,0.125440,0.2207,0.006059
2,0.0051,0.0,0.0,0.0696,0.0170,0.0000,0.0000,0.0357,0.0000,0.0000,0.0,0.0000,0.0000,0.0000,0.0017,0.0,0.0628,0.0000,0.0136,0.0068,0.0696,0.0000,0.0000,0.0000,0.0000,0.0951,0.0594,0.0000,0.0119,0.0,0.0000,0.0000,0.0611,0.1070,0.0051,0.3786,0.0,7523.0,6894.0,0.4331,NaN,16.0,37840050.0,42918486.0,0.0,Lynn University,132657,1,0.163185,0.998617,0.0628,0.3786,0.0051,0.1070,0.0017,0.1766,0.0357,0.0611,0.0696,0.0594,0.0051,0.0051,0.0374,0.186716,0.3786,0.000952
3,0.0000,0.0,0.0,0.0007,0.0019,0.0713,0.0110,0.0024,0.0000,0.0380,0.0,0.0129,0.0014,0.0000,0.3991,0.0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0603,0.0000,0.0543,0.0000,0.0000,0.0,0.0115,0.0081,0.0000,0.0012,0.1706,0.1515,0.0,4778.0,6880.0,0.3651,NaN,22.0,19989561.0,22555029.0,1.0,Eastern Florida State College,132693,1,-0.006074,0.999767,0.1696,0.1515,0.1706,0.0000,0.3991,0.0019,0.0024,0.0196,0.0239,0.0557,0.0000,0.2419,0.4125,0.224948,0.3991,0.054415
4,0.0059,0.0,0.0,0.0014,0.0010,0.0992,0.0023,0.0092,0.0000,0.0119,0.0,0.0155,0.0028,0.0000,0.3970,0.0,0.0000,0.0000,0.0000,0.0025,0.0000,0.0000,0.0000,0.0000,0.0001,0.0000,0.0528,0.0000,0.0001,0.0,0.0035,0.0000,0.0261,0.0086,0.1196,0.2407,0.0,3883.0,6877.0,0.3117,NaN,31.0,44981606.0,45145927.0,1.0,Broward College,132709,1,0.020749,0.999803,0.1112,0.2407,0.1196,0.0001,0.3970,0.0100,0

In [89]:
display(inst_model_df.describe())
display("Feature correlation to target variable",inst_model_df.corr()[targ].sort_values())
display(inst_model_df.info())

,program_percentage_resources,program_percentage_architecture,program_percentage_ethnic_cultural_gender,program_percentage_communication,program_percentage_communications_technology,program_percentage_computer,program_percentage_personal_culinary,program_percentage_education,program_percentage_engineering,program_percentage_engineering_technology,program_percentage_language,program_percentage_family_consumer_science,program_percentage_legal,program_percentage_english,program_percentage_humanities,program_percentage_library,program_percentage_biological,program_percentage_mathematics,program_percentage_military,program_percentage_multidiscipline,program_percentage_parks_recreation_fitness,program_percentage_philosophy_religious,program_percentage_theology_religious_vocation,program_percentage_physical_science,program_percentage_science_technology,program_percentage_psychology,program_percentage_security_law_enforcement,program_percentage_public_administration_social_service,program_percentage_social_science,program_percentage_construction,program_percentage_mechanic_repair_technology,program_percentage_precision_production,program_percentage_transportation,program_percentage_visual_performing,program_percentage_health,program_percentage_business_marketing,program_percentage_history,instructional_expenditure_per_fte,faculty_salary,ft_faculty_rate,program_reporter_programs_offered,student_faculty_ratio,endowment_begin,endowment_end,dolflag,has_endowment,combined_pct_error,weight,pct_stem,pct_business,pct_health,pct_social_science,pct_humanities,pct_arts_comm,pct_education,pct_trades,pct_services,pct_law_security,pct_agriculture,pct_high_roi,pct_low_roi,program_hhi,max_program_share,high_roi_x_concentration
count,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.0,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,117.000000,109.000000,97.000000,198.000000,6.600000e+01,6.600000e+01,196.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000
mean,0.003280,0.000507,0.000152,0.009921,0.004183,0.027252,0.253169,0.009093,0.008770,0.017863,0.000683,0.004504,0.001489,0.002309,0.070909,0.0,0.013964,0.000955,0.000176,0.006207,0.006853,0.000854,0.002491,0.001728,0.001443,0.020120,0.027861,0.002752,0.008540,0.008772,0.046207,0.013601,0.014674,0.013868,0.316221,0.069195,0.001124,7243.400000,7681.974359,0.551708,9.731959,17.560606,1.502573e+08,1.644676e+08,0.448980,0.330000,0.004404,0.996530,0.071974,0.069195,0.316221,0.032536,0.077398,0.023790,0.009093,0.083255,0.264526,0.029350,0.003280,0.352244,0.333171,0.569290,0.654407,0.239629
std,0.013092,0.003184,0.000749,0.023324,0.029102,0.049098,0.393137,0.020643,0.043544,0.061442,0.004426,0.023700,0.004137,0.007101,0.158018,0.0,0.046813,0.003107,0.001801,0.032519,0.026552,0.005277,0.022769,0.006523,0.008783,0.072916,0.064280,0.010187,0.025776,0.037072,0.162643,0.076012,0.062278,0.057076,0.362691,0.119149,0.003869,4856.346435,2103.695822,0.287385,8.312284,8.057148,3.514913e+08,3.799826e+08,0.498664,0.471393,0.094743,0.005492,0.112992,0.119149,0.362691,0.088799,0.161153,0.067465,0.020643,0.206866,0.388904,0.064675,0.013092,0.355005,0.377739,0.355454,0.307604,0.363131
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,294.000000,3813.000000,0.050300,1.000000,2.0000

C:\Users\sebas\AppData\Local\Temp\ipykernel_30332\3989557815.py:2: FutureWarning: The default value of numeric_only in DataFrame.corr is deprecated. In a future version, it will default to False. Select only valid columns or specify the value of numeric_only to silence this warning.
  display("Feature correlation to target variable",inst_model_df.corr()[targ].sort_values())


'Feature correlation to target variable'

has_endowment                                   -0.138519
program_percentage_language                     -0.131757
weight                                          -0.128139
program_percentage_multidiscipline              -0.114163
faculty_salary                                  -0.112820
                                                   ...   
program_reporter_programs_offered                0.183704
program_percentage_mechanic_repair_technology    0.288386
pct_trades                                       0.295942
combined_pct_error                               1.000000
program_percentage_library                            NaN
Name: combined_pct_error, Length: 64, dtype: float64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 66 columns):
 #   Column                                                   Non-Null Count  Dtype  
---  ------                                                   --------------  -----  
 0   program_percentage_resources                             200 non-null    float64
 1   program_percentage_architecture                          200 non-null    float64
 2   program_percentage_ethnic_cultural_gender                200 non-null    float64
 3   program_percentage_communication                         200 non-null    float64
 4   program_percentage_communications_technology             200 non-null    float64
 5   program_percentage_computer                              200 non-null    float64
 6   program_percentage_personal_culinary                     200 non-null    float64
 7   program_percentage_education                             200 non-null    float64
 8   program_percentage_engineering

None

In [90]:
def run_school_corr_screen(merge_df, feature_cols, target_col="combined_pct_error", weight_col="weight"):
    
    df_tmp = merge_df[feature_cols + [target_col, weight_col]].copy()
    df_tmp = df_tmp.dropna(subset=[target_col])
    df_tmp = df_tmp.dropna(subset=feature_cols, how="all")
    
    print(f"Rows after dropna: {len(df_tmp)}")
    print(f"Features: {len(feature_cols)}")
    print(f"Target distribution:\n{df_tmp[target_col].describe().round(4)}\n")

    # winsorize target
    lo, hi = df_tmp[target_col].quantile(0.02), df_tmp[target_col].quantile(0.98)
    df_tmp[target_col] = df_tmp[target_col].clip(lo, hi)

    X = df_tmp[feature_cols].apply(pd.to_numeric, errors="coerce")
    y = df_tmp[target_col]
    w = df_tmp[weight_col]

    # ── correlation screen ───────────────────────────────────────
    corr = X.corrwith(y).abs().sort_values(ascending=False)
    
    print("Top 15 correlations with target:")
    print(corr.head(15).round(4).to_string())
    print(f"\nFeatures with |corr| > 0.10: {(corr > 0.10).sum()}")
    print(f"Features with |corr| > 0.15: {(corr > 0.15).sum()}")
    print(f"Features with |corr| > 0.20: {(corr > 0.20).sum()}")

    # ── RF signal check ──────────────────────────────────────────
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import KFold
    from sklearn.metrics import r2_score
    from sklearn.impute import SimpleImputer

    imputer = SimpleImputer(strategy="median")
    X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rf = RandomForestRegressor(
        n_estimators=100, 
        max_depth=3,        # shallow — only 200 rows
        random_state=42, 
        n_jobs=-1
    )

    fold_r2s = []
    for train_idx, val_idx in kf.split(X_imp):
        X_tr, X_val = X_imp.iloc[train_idx], X_imp.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        w_tr, w_val = w.iloc[train_idx], w.iloc[val_idx]

        rf.fit(X_tr, y_tr, sample_weight=w_tr)
        y_pred = rf.predict(X_val)
        fold_r2s.append(r2_score(y_val, y_pred, sample_weight=w_val))

    print(f"\nRF CV R² (max_depth=3): {np.mean(fold_r2s):.4f} ± {np.std(fold_r2s):.4f}")
    print(f"Fold R²s: {[round(s,3) for s in fold_r2s]}")

    # ── feature importance if R² is positive ────────────────────
    if np.mean(fold_r2s) > 0:
        rf.fit(X_imp, y, sample_weight=w)
        imp_df = pd.DataFrame({
            "feature": feature_cols,
            "importance": rf.feature_importances_
        }).sort_values("importance", ascending=False)
        print("\nTop 10 feature importances:")
        print(imp_df.head(10).to_string(index=False))

    return corr


# ── RUN ─────────────────────────────────────────────────────────
feature_cols = [c for c in inst_model_df.columns if c not in [
    "unit_id", "school_name", "weight", "combined_pct_error",
    "avg_rank_stability", "total_pred", "median_error",
    "total_count", "total_count_1", "total_count_4", "total_count_5"
]]

corr_results = run_school_corr_screen(inst_model_df, feature_cols)

Rows after dropna: 200
Features: 62
Target distribution:
count    200.0000
mean       0.0044
std        0.0947
min       -0.4613
25%       -0.0371
50%       -0.0040
75%        0.0367
max        0.4165
Name: combined_pct_error, dtype: float64

Top 15 correlations with target:
pct_trades                                       0.2802
program_percentage_mechanic_repair_technology    0.2617
program_reporter_programs_offered                0.1900
has_endowment                                    0.1639
program_percentage_language                      0.1506
program_percentage_precision_production          0.1455
faculty_salary                                   0.1393
program_percentage_multidiscipline               0.1311
pct_low_roi                                      0.1270
program_percentage_philosophy_religious          0.1190
program_percentage_biological                    0.1160
program_percentage_family_consumer_science       0.1119
ft_faculty_rate                                  0.1


RF CV R² (max_depth=3): -0.0247 ± 0.1345
Fold R²s: [0.001, -0.229, 0.12, 0.107, -0.123]


In [91]:
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score

# Reproducibility
RANDOM_STATE = 42

In [92]:
# -------------------
# 1. Define target
# -------------------

# Drop target and target-like columns from X
feature_cols = [c for c in inst_model_df.columns if c not in ["unit_id", targ, "weight",'credential_level','code','avg_rank_stability']]

X = inst_model_df[feature_cols]

y = np.log1p(inst_model_df[targ].abs())
# y = y.clip(upper=y.quantile(0.9))

w = inst_model_df["weight"]

# Force predictors numeric
X = X.apply(pd.to_numeric, errors="coerce")

# -------------------
# 2. Train/test split
# -------------------
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.2, random_state=RANDOM_STATE
)

# -------------------
# 4. Recompute numeric columns
# -------------------
num_cols = X_train.columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols)
])

In [94]:
len(y)

200

In [ ]:
ridge_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", Ridge())
])

# ridge_model = TransformedTargetRegressor(
#     regressor=ridge_pipe,
#     transformer=PowerTransformer(method="yeo-johnson", standardize=False)
# )

ridge_param_grid = {
    "reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0, 100.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

ridge_grid = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=ridge_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

ridge_grid.fit(
    X_train,
    y_train,
    reg__sample_weight=w_train
)

ridge_best = ridge_grid.best_estimator_
ridge_preds = ridge_best.predict(X_test)

print("RIDGE")
print("Best params:", ridge_grid.best_params_)
print("Best CV MAE:", round(-ridge_grid.best_score_, 4))
print("Test MAE (vs unclipped y_test):", round(mean_absolute_error(y_test, ridge_preds), 4))
print("Test R2 (vs unclipped y_test):", round(r2_score(y_test, ridge_preds), 4))

# feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
# print(feature_names)

Fitting 5 folds for each of 7 candidates, totalling 35 fits
RIDGE
Best params: {'reg__alpha': 100.0}
Best CV MAE: 0.047
Test MAE (vs unclipped y_test): 0.0476
Test R2 (vs unclipped y_test): -0.5164


In [114]:
# check these immediately
print(X_train[stable_features].shape)
print(X_test[stable_features].shape)
print(y_train.shape, y_test.shape)
print(w_train.shape)

# check if indices are reset
print(X_train.index[:5])
print(y_train.index[:5])
print(w_train.index[:5])

(160, 11)
(40, 11)
(160,) (40,)
(160,)
Int64Index([79, 197, 38, 24, 122], dtype='int64')
Int64Index([79, 197, 38, 24, 122], dtype='int64')
Int64Index([79, 197, 38, 24, 122], dtype='int64')


In [103]:
from sklearn.linear_model import Lasso


lasso_param_grid = {
    "reg__alpha": np.logspace(-4, 1, 50)
}

lasso_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", Lasso(max_iter=10000, random_state=RANDOM_STATE))
])

lasso_grid = GridSearchCV(
    estimator=lasso_pipe,
    param_grid=lasso_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

lasso_grid.fit(
    X_train,
    y_train,
    reg__sample_weight=w_train
)

lasso_best = lasso_grid.best_estimator_
lasso_preds = lasso_best.predict(X_test)

print("Test MAE (vs unclipped y_test):", round(mean_absolute_error(y_test, lasso_preds), 4))
print("Test R2 (vs unclipped y_test):", round(r2_score(y_test, lasso_preds), 4))

feature_names = lasso_best.named_steps["preprocessor"].get_feature_names_out()

coef_df = pd.DataFrame({
    "feature": [f.replace("num__", "") for f in feature_names],
    "coefficient": lasso_best.named_steps["reg"].coef_
}).sort_values("coefficient", key=abs, ascending=False)

print(coef_df[coef_df["coefficient"] != 0].to_string(index=False))
print(f"\nFeatures selected: {(coef_df['coefficient'] != 0).sum()} of {len(coef_df)}")
print(f"Lasso alpha chosen: {lasso_best.named_steps['reg'].alpha:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Test MAE (vs unclipped y_test): 0.0409
Test R2 (vs unclipped y_test): 0.1743
                                       feature  coefficient
 program_percentage_mechanic_repair_technology     0.008098
                                      pct_stem    -0.005212
                                    pct_trades     0.002814
                                 has_endowment    -0.002378
                   program_percentage_computer    -0.001495
         program_percentage_business_marketing    -0.001343
                   program_percentage_military     0.001058
                      program_percentage_legal    -0.000530
program_percentage_theology_religious_vocation     0.000251
                                  pct_business    -0.000044

Features selected: 10 of 62
Lasso alpha chosen: 0.0069


In [110]:
from sklearn.linear_model import LassoCV
from sklearn.pipeline import Pipeline

# stability check — do the same features survive across seeds?
for seed in [0, 7, 13, 21, 99]:
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("reg", LassoCV(cv=5, random_state=seed, max_iter=50000))
    ])
    pipe.fit(X_train, y_train, reg__sample_weight=w_train)

    feature_names = pipe.named_steps["preprocessor"].get_feature_names_out()
    selected = [
        f.replace("num__", "")
        for f, c in zip(feature_names, pipe.named_steps["reg"].coef_)
        if c != 0
    ]
    print(f"Seed {seed:3d}: {selected}")

Seed   0: ['program_percentage_computer', 'program_percentage_engineering_technology', 'program_percentage_legal', 'program_percentage_military', 'program_percentage_theology_religious_vocation', 'program_percentage_social_science', 'program_percentage_mechanic_repair_technology', 'program_percentage_business_marketing', 'has_endowment', 'pct_stem', 'pct_trades']
Seed   7: ['program_percentage_computer', 'program_percentage_engineering_technology', 'program_percentage_legal', 'program_percentage_military', 'program_percentage_theology_religious_vocation', 'program_percentage_social_science', 'program_percentage_mechanic_repair_technology', 'program_percentage_business_marketing', 'has_endowment', 'pct_stem', 'pct_trades']
Seed  13: ['program_percentage_computer', 'program_percentage_engineering_technology', 'program_percentage_legal', 'program_percentage_military', 'program_percentage_theology_religious_vocation', 'program_percentage_social_science', 'program_percentage_mechanic_repair

In [113]:
from sklearn.linear_model import RidgeCV

# check the STEM story
stable_features = [
    'program_percentage_computer',
    'program_percentage_engineering_technology',
    'program_percentage_legal',
    'program_percentage_military',
    'program_percentage_theology_religious_vocation',
    'program_percentage_social_science',
    'program_percentage_mechanic_repair_technology',
    'program_percentage_business_marketing',
    'has_endowment',
    'pct_stem',
    'pct_trades'
]

# Refit Ridge on stable features only — Ridge is better than Lasso
# for final model when you've already done feature selection
pipe_final = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", RidgeCV(alphas=np.logspace(-2, 3, 50), cv=5))
])

pipe_final.fit(X_train[stable_features], y_train, ridge__sample_weight=w_train)

y_pred = pipe_final.predict(X_test[stable_features])
print("Final R²:", round(r2_score(y_test, y_pred), 4))
print("Final MAE:", round(mean_absolute_error(y_test, y_pred), 4))

# coefficients
coef_df = pd.DataFrame({
    "feature": stable_features,
    "coefficient": pipe_final.named_steps["ridge"].coef_
}).sort_values("coefficient", ascending=False)
print(coef_df.to_string(index=False))

Final R²: -0.6316
Final MAE: 0.0495
                                       feature  coefficient
 program_percentage_mechanic_repair_technology     0.007431
                                    pct_trades     0.007155
                   program_percentage_military     0.005751
program_percentage_theology_religious_vocation     0.005441
                                      pct_stem    -0.003294
             program_percentage_social_science    -0.003536
                      program_percentage_legal    -0.003573
         program_percentage_business_marketing    -0.003918
                                 has_endowment    -0.004477
                   program_percentage_computer    -0.004525
     program_percentage_engineering_technology    -0.005652


In [116]:
# Step 1: try without sample weights at all
pipe_final.fit(X_train[stable_features], y_train)
y_pred = pipe_final.predict(X_test[stable_features])
print("R² no weights:", round(r2_score(y_test, y_pred), 4))

# Step 2: try plain Ridge instead of RidgeCV
from sklearn.linear_model import Ridge

pipe_plain = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])
pipe_plain.fit(X_train[stable_features], y_train)
y_pred = pipe_plain.predict(X_test[stable_features])
print("R² plain Ridge alpha=1:", round(r2_score(y_test, y_pred), 4))

# Step 3: check if test set is just unlucky
from sklearn.model_selection import cross_val_score
cv_r2 = cross_val_score(pipe_plain, 
                         inst_model_df[stable_features], 
                         inst_model_df["combined_pct_error"],
                         cv=5, scoring="r2")
print("CV R²:", cv_r2.mean().round(4), "±", cv_r2.std().round(4))
print("Folds:", [round(s,3) for s in cv_r2])

R² no weights: -0.6407
R² plain Ridge alpha=1: -1.6635
CV R²: 0.0234 ± 0.0928
Folds: [-0.049, 0.136, 0.131, -0.015, -0.086]


In [118]:
# Use all 193 rows for CV instead of holding out a test set
# With this sample size a test split is too noisy to be meaningful

from sklearn.model_selection import RepeatedKFold

# Repeated CV averages over many different splits — much more stable
rkf = RepeatedKFold(n_splits=5, n_repeats=20, random_state=42)

cv_r2 = cross_val_score(
    pipe_final,
    inst_model_df[stable_features],
    inst_model_df["combined_pct_error"],
    cv=rkf,
    scoring="r2",
    fit_params={"ridge__sample_weight": merge_df["weight"]}
)

print(f"Repeated CV R² (5x20): {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")

Repeated CV R² (5x20): -0.0596 ± 0.2453


In [119]:
enet_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", ElasticNet(max_iter=100000))
])

# enet_model = TransformedTargetRegressor(
#     regressor=enet_pipe,
#     transformer=PowerTransformer(method="yeo-johnson", standardize=False)
# )

enet_param_grid = {
    "reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0],
    "reg__l1_ratio": [0.005,0.1, 0.3, 0.5, 0.7, 0.9]
}

enet_grid = GridSearchCV(
    estimator=enet_pipe,
    param_grid=enet_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

enet_grid.fit(
    X_train, 
    y_train,
    reg__sample_weight=w_train
)

enet_best = enet_grid.best_estimator_
enet_preds = enet_best.predict(X_test)

print("\nELASTIC NET")
print("Best params:", enet_grid.best_params_)
print("Best CV MAE:", round(-enet_grid.best_score_, 4))
print("Test MAE (vs unclipped y_test):", round(mean_absolute_error(y_test, enet_preds), 4))
print("Test R2 (vs unclipped y_test):", round(r2_score(y_test, enet_preds), 4))

Fitting 5 folds for each of 36 candidates, totalling 180 fits

ELASTIC NET
Best params: {'reg__alpha': 1.0, 'reg__l1_ratio': 0.005}
Best CV MAE: 0.0446
Test MAE (vs unclipped y_test): 0.0431
Test R2 (vs unclipped y_test): 0.0802


In [120]:
from sklearn.ensemble import HistGradientBoostingRegressor
def run_hgbr():
    model = HistGradientBoostingRegressor( 
        max_depth=3,
        learning_rate=0.05,
        max_iter=200,
        random_state=42 
        ) 

    model.fit(
        X_train, 
        y_train,
        sample_weight=w_train
    ) 
    preds = model.predict(X_test) 
    print("MAE:", mean_absolute_error(y_test, preds)) 
    print("R2:", r2_score(y_test, preds))
run_hgbr()

MAE: 0.03680259813736687
R2: 0.25343221110504643


In [121]:
def run_hgbr_2():
    hgbr_model_2 = Pipeline([
        ('preprocessor',preprocessor),
        ('model',HistGradientBoostingRegressor(
        max_depth=5,              # allow more interactions
        learning_rate=0.01,       # slower learning
        max_iter=750,             # more trees
        min_samples_leaf=10,      # regularization
        l2_regularization=2.0,    # stabilize
        random_state=42
        ))
    ])
  

    hgbr_model_2.fit(
        X_train, 
        y_train,
        model__sample_weight=w_train
    ) 

    preds_2 = hgbr_model_2.predict(X_test) 
    print("MAE:", mean_absolute_error(y_test, preds_2)) 
    print("R2:", r2_score(y_test, preds_2))
run_hgbr_2()

MAE: 0.0418632852065723
R2: 0.08320838630338734


In [122]:
from sklearn.model_selection import cross_val_score

# bin target into quantiles
y_bins = pd.qcut(y, q=5, labels=False, duplicates="drop")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_hgbr_model = Pipeline([
        ('preprocessor',preprocessor),
        ('model',HistGradientBoostingRegressor( 
        max_depth=3,
        learning_rate=0.05,
        max_iter=200,
        random_state=42 
        ))
    ])

scores = cross_val_score(
    cv_hgbr_model,
    X,
    y,
    cv=cv.split(X, y_bins),
    scoring="r2"
)

print('HistGradientBoostingRegressor\n',scores)
print("mean:", scores.mean())
print("std:", scores.std())

HistGradientBoostingRegressor
 [-0.066535    0.03641682  0.19890449 -0.416823   -0.00445811]
mean: -0.05049896141933215
std: 0.20313869728746575


In [123]:
y_shuffled = y.sample(frac=1, random_state=42).reset_index(drop=True)

cv_hgbr_model.fit(X_train, y_shuffled.loc[X_train.index], model__sample_weight=w_train)
preds = cv_hgbr_model.predict(X_test)

print("R2 shuffled:", r2_score(y_test, preds))

R2 shuffled: -0.18905159396498195


In [124]:
# bin target into quantiles
y_bins = pd.qcut(y, q=5, labels=False, duplicates="drop")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    ridge_pipe,
    X,
    y,
    cv=cv.split(X, y_bins),
    scoring="r2",
    fit_params={"reg__sample_weight": w}
)

print(scores)
print("mean:", scores.mean())
print("std:", scores.std())

[-2.23007631 -0.90811337 -0.04470186 -0.63937749 -1.68354585]
mean: -1.1011629779146286
std: 0.7720413469057782


In [125]:
from xgboost import XGBRegressor

cv_xgb_model = Pipeline([
        ('preprocessor',preprocessor),
        ('model',XGBRegressor(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            random_state=42
        ))
    ])

scores = cross_val_score(
    cv_xgb_model,
    X,
    y,
    cv=cv.split(X, y_bins),
    scoring="r2",
    fit_params={"model__sample_weight": w}
)

print("XGBRegressor")
print(scores)
print("mean:", scores.mean())
print("std:", scores.std())

XGBRegressor
[-0.48600063  0.02861512  0.3277264  -0.92091734 -0.08420514]
mean: -0.22695631852908343
std: 0.43406299636632445


In [126]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score



# -------------------
# 1. Target (top vs bottom)
# -------------------
y_raw = np.log1p(inst_model_df[targ].abs())
# y_raw = y.clip(upper=y.quantile(0.95))
# y_raw = inst_model_df[targ].copy()

low = y_raw.quantile(0.2)
high = y_raw.quantile(0.8)

mask = (y_raw <= low) | (y_raw >= high)

# -------------------
# 2. Use ALL columns except target
# -------------------
X_clf = inst_model_df.drop(columns=["unit_id", targ, "weight",'credential_level','code','avg_rank_stability'], errors="ignore").loc[mask].copy()
y_clf = (y_raw.loc[mask] >= high).astype(int)
w_clf = inst_model_df['weight'].loc[mask].copy()

# force everything numeric
X_clf = X_clf.apply(pd.to_numeric, errors="coerce")

# -------------------
# 3. Train/test split
# -------------------
X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X_clf,
    y_clf,
    w_clf,
    test_size=0.3,
    random_state=RANDOM_STATE,
    stratify=y_clf
)

# -------------------
# 4. Preprocessing (CRITICAL)
# -------------------
num_cols = X_train.columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols)
])

In [127]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# -------------------
# 5. Model
# -------------------
clf = Pipeline([
    ("preprocessor", preprocessor),
    ("logit", LogisticRegression(max_iter=10000))
])

# -------------------
# 6. Fit
# -------------------
clf.fit(
    X_train, 
    y_train,
    logit__sample_weight=w_train
)

# -------------------
# 7. Evaluate
# -------------------
preds = clf.predict(X_test)
probs = clf.predict_proba(X_test)[:, 1]

print(y_clf.value_counts())
print("Accuracy:", round(accuracy_score(y_test, preds), 4))
print("ROC AUC:", round(roc_auc_score(y_test, probs), 4))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, preds))
print("\nClassification Report:\n", classification_report(y_test, preds))

"""
removed bad columns from X "unit_id", targ, "weight",'credential_level','code','avg_rank_stability'
Accuracy: 0.6667
ROC AUC: 0.7333

Confusion Matrix:
 [[10  5]
 [ 5 10]]

"""

1    40
0    40
Name: combined_pct_error, dtype: int64
Accuracy: 0.5833
ROC AUC: 0.6181

Confusion Matrix:
 [[7 5]
 [5 7]]

Classification Report:
               precision    recall  f1-score   support

           0       0.58      0.58      0.58        12
           1       0.58      0.58      0.58        12

    accuracy                           0.58        24
   macro avg       0.58      0.58      0.58        24
weighted avg       0.58      0.58      0.58        24



'\nremoved bad columns from X "unit_id", targ, "weight",\'credential_level\',\'code\',\'avg_rank_stability\'\nAccuracy: 0.6667\nROC AUC: 0.7333\n\nConfusion Matrix:\n [[10  5]\n [ 5 10]]\n\n'